# Chapter 07: Contexts, Expressions and Conditionals (notebook edition)

## Learning Objectives

- Replay real expression results
- Predict truthiness and coercion
- Read outcome vs conclusion

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

GHA_MODE = os.environ.get("GHA_MODE", "fixture")  # "fixture" | "live"
GHA_REPO = os.environ.get("GHA_REPO", "Friend09/practice_git_intro_to_github_actions")
print("mode:", GHA_MODE, "| repo:", GHA_REPO)

## 1. Replay the real engine

Two real runs evaluated the same 32 expressions. Our evaluator should reproduce every printed value. You should see `64 checked`.

In [ ]:
import json, re
from intro_gha.expr import evaluate, stringify
ECHO = re.compile(r'echo "REPORT (\w+)=\$\{\{ (.+) \}\}"$')
text = (ROOT/".github"/"workflows"/"ch07-expressions.yml").read_text()
exprs = {m[1]: m[2] for l in text.splitlines() if (m := ECHO.search(l.strip()))}
def ctx(inputs):
    return {"github": {"event_name": "workflow_dispatch", "ref_name": "main", "event": {"pull_request": {"title": None}}},
            "runner": {"os": "Linux"}, "env": {"FLAG_TEXT": "false", "ZERO_TEXT": "0", "EMPTY": ""},
            "inputs": inputs, "job": {"status": "success"}}
checked = 0
for run, inputs in {"defaults": {"name": "", "flag": False}, "inputs": {"name": "Tally", "flag": True}}.items():
    data = json.loads((ROOT/"fixtures"/f"ctx_ch07_{run}.json").read_text())
    for key, observed in data["reports"]["literals"].items():
        if key in exprs:
            assert stringify(evaluate(exprs[key], ctx(inputs))) == observed, key
            checked += 1
print(checked, "checked")
assert checked == 64

## 2. The truthiness table

The strings `'false'` and `'0'` are truthy; the boolean input behaves.

In [ ]:
c = ctx({"name": "", "flag": False})
for e in ["'false'", "'0'", "0", "''", "null", "env.FLAG_TEXT", "inputs.flag"]:
    print(f"{e:16}", stringify(evaluate(e + " && 'truthy' || 'falsy'", c)))
assert stringify(evaluate("env.FLAG_TEXT && 'truthy' || 'falsy'", c)) == "truthy"
assert stringify(evaluate("inputs.flag && 'truthy' || 'falsy'", c)) == "falsy"

## 3. Comparison surprises

Two strings compare as text; a string against a number compares as numbers.

In [ ]:
for e, want in [("'10' > '9'", "false"), ("'10' > 9", "true"), ("'3.10' > '3.9'", "false"), ("null == ''", "true"), ("'ABC' == 'abc'", "true")]:
    got = stringify(evaluate(e, c)); print(f"{e:18} {got}")
    assert got == want

## 4. No arithmetic, and hashFiles

GitHub expressions have no `+`; `hashFiles` of one file is `sha256(sha256(bytes))`.

In [ ]:
from intro_gha.expr import ExprError
try:
    evaluate("1 + 1", c)
except ExprError as e:
    print("rejected:", e)
import hashlib
b = (ROOT/"sandbox"/"tally"/"tally"/"add.py").read_bytes()
mine = hashlib.sha256(hashlib.sha256(b).digest()).hexdigest()
real = json.loads((ROOT/"fixtures"/"ctx_ch07_defaults.json").read_text())["reports"]["matrixed (3.10)"]["hash"]
print(mine == real)
assert mine == real

## 5. outcome vs conclusion

A step with `continue-on-error` that fails has outcome `failure` and conclusion `success`, so the `failure()` guard never ran.

In [ ]:
s = json.loads((ROOT/"fixtures"/"ctx_ch07_defaults.json").read_text())["reports"]["status"]
print(s)
assert (s["flaky_outcome"], s["flaky_conclusion"]) == ("failure", "success") and "failure_step_ran" not in s

## Takeaways & Next Steps

- `'false'` is truthy; compare strings explicitly.
- No arithmetic; status functions only in `if:`.
- Next: Chapter 08, variables, secrets and environments.

## Chapter Link

Read: `learning_modules/chapter_07_contexts_expressions.md`